# 23_GSE232240_therapy_analysis.ipynb

**Thesis Methods section: 4.1.8 (therapy-associated analysis)** — produces Figure S4.

GSE232240 (IMCISION neoadjuvant immune checkpoint blockade, head and neck squamous cell
carcinoma) is analysed separately from the integrated atlas, as an independent
therapy-associated cohort.

**Reads:** GSM7324294_Count_data_IMCISION.txt and GSM7324295_Meta_data_IMCISION.txt (GEO).

**Writes:** GSE232240_IMCISION.h5ad, GSE232240_IMCISION_CD8.h5ad.

**What it does:** restricts to CD8 cells via the author-provided `cell_type` annotation,
inspects TRAC, CD3D, CD8A, CD8B, NKG7, MS4A1 and LYZ as a marker verification step, and
drops patients with fewer than 100 CD8 cells from patient-level statistics. Exhaustion and
proliferation are scored with `scanpy.tl.score_genes` using the nine-gene panels of Methods
4.1.7 (deposited in `panels/`), z-normalised within the dataset. Prolif-Tex and
NonProlif-Tex are assigned with the same Q75/Q75 and Q75/Q25 rules, thresholds recalculated
within this cohort. Per patient and timepoint, configuration prevalence and continuous
proliferation summaries (mean and 90th percentile) are computed; post-minus-pre changes are
compared between responders and non-responders with two-sided Mann-Whitney U tests, with
Cliff's delta as effect size.

**Cell order:** the classification and patient-summary blocks (Blocks 17 and 18) are placed
immediately after module scoring, so the notebook runs top to bottom. Percentile thresholds
are computed from the same object either way — nothing between scoring and classification
subsets or reassigns `adata_cd8`.

Input data are not included in this repository. Set `DATA_ROOT` below to a local folder
holding the GEO downloads; see `README.md` for accessions.


In [ ]:
# Root folder for input data (NOT included in this repository).
import os
DATA_ROOT = os.environ.get("DATA_ROOT", "data")


## GSE232240 (IMCISION)

Load counts and metadata from the deposited text files, build the AnnData object, subset to CD8, and run the therapy-associated analysis.

**The deposited matrix holds decimal, log-normalised-like values, not raw integer UMI counts.** Do not apply `sc.pp.normalize_total` or `sc.pp.log1p` to it.

In [ ]:
# -----------------------------
# Block 0 — Imports
# -----------------------------
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy import sparse

In [ ]:
# -----------------------------
# Block 1 — Paths (EDIT)
# -----------------------------
counts_path = f"{DATA_ROOT}/scVI/6 GSE232240/GSM7324294_Count_data_IMCISION.txt"
meta_path   = f"{DATA_ROOT}/scVI/6 GSE232240/GSM7324295_Meta_data_IMCISION.txt"

out_h5ad      = f"{DATA_ROOT}/scVI/6 GSE232240/GSE232240_IMCISION.h5ad"
out_cd8_h5ad  = f"{DATA_ROOT}/scVI/6 GSE232240/GSE232240_IMCISION_CD8.h5ad"

print("Counts exists:", os.path.exists(counts_path))
print("Meta exists:", os.path.exists(meta_path))

In [ ]:
# -----------------------------
# Block 2 — Load metadata
# -----------------------------
meta = pd.read_csv(meta_path, sep="\t")
assert "cell_id" in meta.columns, "No 'cell_id' column found in metadata."
meta = meta.set_index("cell_id")
print("Meta loaded:", meta.shape)
print("Meta columns:", list(meta.columns))

In [ ]:
# -----------------------------
# Block 3 — Read header (cell IDs)
# -----------------------------
header = pd.read_csv(counts_path, sep="\t", nrows=0)
cell_ids = header.columns[1:]  # first column = gene names

print("Cells in count matrix (header):", len(cell_ids))
print("Cells in metadata:", meta.shape[0])
print("Overlap:", len(meta.index.intersection(cell_ids)))

In [ ]:
# -----------------------------
# Block 4 — Stream counts in chunks → sparse matrix
#   Output: X_cells_by_genes (cells x genes), genes list
# -----------------------------
chunk_size = 500  # raise to 1000–2000 if RAM allows
blocks = []
genes = []

reader = pd.read_csv(
    counts_path,
    sep="\t",
    index_col=0,        # gene names in first column
    chunksize=chunk_size
)

for i, chunk in enumerate(reader):
    if i % 20 == 0:
        print(f"chunk {i}, shape={chunk.shape}")
    genes.extend(chunk.index.astype(str).tolist())
    blocks.append(sparse.csr_matrix(chunk.values))

X_genes_by_cells = sparse.vstack(blocks, format="csr")
del blocks
gc.collect()

X_cells_by_genes = X_genes_by_cells.T.tocsr()
del X_genes_by_cells
gc.collect()

print("Final sparse shape (cells x genes) BEFORE fix:", X_cells_by_genes.shape)

In [ ]:
# -----------------------------
# Block 5 — Fix occasional +1 cell mismatch vs header
# -----------------------------
n_cells_header = len(cell_ids)
if X_cells_by_genes.shape[0] != n_cells_header:
    print(f"⚠️ Mismatch: matrix has {X_cells_by_genes.shape[0]} cells but header has {n_cells_header}. Truncating.")
    X_cells_by_genes = X_cells_by_genes[:n_cells_header, :]

print("Final sparse shape (cells x genes) AFTER fix:", X_cells_by_genes.shape)

In [ ]:
# -----------------------------
# Block 6 — Build AnnData + attach metadata
# -----------------------------
adata = sc.AnnData(X_cells_by_genes)
adata.obs_names = pd.Index(cell_ids.astype(str), dtype=str)
adata.var_names = pd.Index(genes, dtype=str)

# Align metadata to adata.obs_names
adata.obs = meta.reindex(adata.obs_names)

missing_all = int(adata.obs.isna().all(axis=1).sum())
print("Cells missing ALL metadata:", missing_all)
print(adata)

In [ ]:
# -----------------------------
# Block 7 — Quick matrix sanity check (raw vs log-like)
#   We expect DECIMALS here (log-like), so do NOT normalize/log.
# -----------------------------
sub = adata.X[:5, :5]
if sp.issparse(sub):
    sub = sub.toarray()
print("\nMatrix 5x5 preview:\n", sub)

# -----------------------------
# Block 8 — Save .h5ad (so you never re-parse the TXT)
# -----------------------------
adata.write_h5ad(out_h5ad)
print("Saved:", out_h5ad)

In [ ]:
# -----------------------------
# Block 9 — Cohort summary (all CD45+ tumour immune)
# -----------------------------
print("\nPatients:", adata.obs["patient"].nunique() if "patient" in adata.obs.columns else "no patient column")
for col in ["timepoint", "response"]:
    if col in adata.obs.columns:
        print(f"\n{col} counts:\n", adata.obs[col].value_counts(dropna=False))

In [ ]:
# -----------------------------
# Block 10 — Helper functions (safe sparse handling)
# -----------------------------
def gene_vec(adata_obj, gene):
    """Return 1D numpy array of expression for a single gene."""
    v = adata_obj[:, gene].X
    if sp.issparse(v):
        return np.asarray(v.toarray()).ravel()
    return np.asarray(v).ravel()

def frac_pos(adata_obj, gene):
    """Fraction of cells with expression > 0 for a gene."""
    if gene not in adata_obj.var_names:
        return np.nan
    v = gene_vec(adata_obj, gene)
    return float((v > 0).mean())

In [ ]:
# -----------------------------
# Block 11 — Rough CD8 estimate (marker-based; quick QC)
# -----------------------------
for g in ["CD3D", "CD3E", "TRAC", "CD8A", "CD8B", "NKG7", "MS4A1", "LYZ"]:
    print(g, g in adata.var_names)

cd8_mask_rough = (gene_vec(adata, "CD3D") > 0) & (gene_vec(adata, "CD8A") > 0)
print("\nRough CD3D+CD8A+ cells:", int(cd8_mask_rough.sum()))

if "patient" in adata.obs.columns:
    print("\nCD8 per patient (top 20):")
    print(adata.obs.loc[cd8_mask_rough, "patient"].value_counts().head(20))

In [ ]:
# -----------------------------
# Block 12 — CD8 subset (metadata-driven; preferred)
#   Uses adata.obs['cell_type'] provided by authors
# -----------------------------
assert "cell_type" in adata.obs.columns, "No 'cell_type' column found in metadata."

print("\ncell_type value counts (top 50):")
print(adata.obs["cell_type"].value_counts().head(50))

# Auto-select labels containing 'CD8' (edit manually if needed)
cd8_labels = [x for x in adata.obs["cell_type"].unique() if "CD8" in str(x)]
print("\nAuto CD8 labels:", cd8_labels)

adata_cd8 = adata[adata.obs["cell_type"].isin(cd8_labels)].copy()
print("\nCD8 subset:", adata_cd8)

print("\nMarker fractions in CD8 subset:")
for g in ["TRAC", "CD3D", "CD8A", "CD8B", "NKG7", "MS4A1", "LYZ"]:
    print(g, frac_pos(adata_cd8, g))

In [ ]:
# -----------------------------
# Block 13 — Confirm CD8 exists in all 4 groups (pre/post × RE/NR)
# -----------------------------
print("\nCD8 cells by timepoint × response:")
print(adata_cd8.obs.groupby(["timepoint", "response"]).size())

In [ ]:
# -----------------------------
# Block 14 — Filter tiny patients for stats robustness
#   Keep >=100 CD8 cells per patient for patient-level analyses
# -----------------------------
counts_per_patient = adata_cd8.obs["patient"].value_counts()
valid_patients = counts_per_patient[counts_per_patient >= 100].index
dropped_patients = sorted(list(set(adata_cd8.obs["patient"].unique()) - set(valid_patients)))

adata_cd8_stats = adata_cd8[adata_cd8.obs["patient"].isin(valid_patients)].copy()

print("\nCD8 cells (all):", adata_cd8.n_obs)
print("CD8 cells (patients >=100):", adata_cd8_stats.n_obs)
print("Dropped patients (<100 CD8):", dropped_patients)

In [ ]:
# -----------------------------
# Block 14.5 — Fix non-finite values (inf/-inf/NaN) in X
# -----------------------------
import numpy as np
import scipy.sparse as sp

def sanitize_X_inplace(adata_obj):
    X = adata_obj.X
    if sp.issparse(X):
        # Work on stored non-zero entries only
        data = X.data
        bad = ~np.isfinite(data)
        n_bad = int(bad.sum())
        if n_bad > 0:
            print(f"⚠️ Found {n_bad} non-finite values in sparse X.data. Replacing with 0.")
            data[bad] = 0.0
            X.eliminate_zeros()
        else:
            print("✅ No non-finite values found in sparse X.data.")
    else:
        bad = ~np.isfinite(X)
        n_bad = int(bad.sum())
        if n_bad > 0:
            print(f"⚠️ Found {n_bad} non-finite values in dense X. Replacing with 0.")
            X[bad] = 0.0
        else:
            print("✅ No non-finite values found in dense X.")
    adata_obj.X = X

sanitize_X_inplace(adata_cd8)

In [ ]:
# -----------------------------
# Block 15 — CD8 UMAP + clustering (ROBUST: no HVGs)
#   Fixes Scanpy HVG error by:
#   1) clipping extreme values in sparse X.data
#   2) PCA/Neighbors/UMAP directly
# -----------------------------
import numpy as np
import scipy.sparse as sp

def clip_sparse_inplace(adata_obj, clip_max=20.0):
    """Clip extreme values in sparse matrix non-zeros to avoid overflow."""
    X = adata_obj.X
    if not sp.issparse(X):
        # dense fallback
        X = np.clip(X, -clip_max, clip_max)
        adata_obj.X = X
        return
    X = X.tocsr()
    X.data = np.clip(X.data, -clip_max, clip_max)
    X.eliminate_zeros()
    adata_obj.X = X

# Clip extremes (log-like expression should not need values > ~20)
clip_sparse_inplace(adata_cd8, clip_max=20.0)

print("After clipping — nnz:", adata_cd8.X.nnz,
      "min:", float(adata_cd8.X.data.min()) if adata_cd8.X.nnz else 0.0,
      "max:", float(adata_cd8.X.data.max()) if adata_cd8.X.nnz else 0.0)

# PCA/UMAP
sc.pp.pca(adata_cd8, n_comps=50)
sc.pp.neighbors(adata_cd8, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_cd8)
sc.tl.leiden(adata_cd8, resolution=0.6)

sc.pl.umap(adata_cd8, color=["leiden", "timepoint", "response"], wspace=0.4)

In [ ]:
# -----------------------------
# Block 16 — Module scoring (Exhaustion + Proliferation) + within-dataset z-normalisation
#   Mirrors thesis methods:
#   - Score programmes independent of clusters
#   - Z-normalise within dataset to minimise magnitude differences
# -----------------------------

# Curated gene sets from thesis methods
exhaustion_signature = ["PDCD1", "LAYN", "HAVCR2", "CTLA4", "TOX", "TIGIT", "LAG3", "CXCL13", "ENTPD1"]
proliferation_signature = ["MKI67", "TOP2A", "PCNA", "HMGB2", "CDC20", "UBE2C", "BIRC5", "AURKB", "CCNB1"]

# Keep only genes present (prevents warnings and improves comparability)
exhaustion_signature = [g for g in exhaustion_signature if g in adata_cd8.var_names]
proliferation_signature = [g for g in proliferation_signature if g in adata_cd8.var_names]

print("Exhaustion genes present:", exhaustion_signature)
print("Proliferation genes present:", proliferation_signature)

# Score programmes (Scanpy analogue of module scoring)
sc.tl.score_genes(adata_cd8, exhaustion_signature, score_name="exhaustion_score_raw")
sc.tl.score_genes(adata_cd8, proliferation_signature, score_name="prolif_score_raw")

# Z-normalise within THIS dataset (dataset-level standardisation)
def zscore_series(x):
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    if sd == 0 or np.isnan(sd):
        return np.zeros_like(x)
    return (x - mu) / sd

adata_cd8.obs["exhaustion_score_z"] = zscore_series(adata_cd8.obs["exhaustion_score_raw"])
adata_cd8.obs["prolif_score_z"] = zscore_series(adata_cd8.obs["prolif_score_raw"])

# Visual check
sc.pl.umap(
    adata_cd8,
    color=["exhaustion_score_z", "prolif_score_z"],
    wspace=0.3
)


In [ ]:
# -----------------------------
# Block 17 — Percentile-based classification of exhausted CD8 into Prolif-Tex vs Non-Prolif-Tex
#   Mirrors thesis methods:
#   - Prolif-Tex: Exhaustion >= Q75 AND Prolif >= Q75
#   - Non-Prolif-Tex: Exhaustion >= Q75 AND Prolif <= Q25
#   - others: unlabelled
# -----------------------------

# Compute thresholds across THIS CD8 tumour cohort
exh_q75 = adata_cd8.obs["exhaustion_score_z"].quantile(0.75)
pro_q75 = adata_cd8.obs["prolif_score_z"].quantile(0.75)
pro_q25 = adata_cd8.obs["prolif_score_z"].quantile(0.25)

print("Thresholds:")
print("Exhaustion Q75:", float(exh_q75))
print("Prolif Q75:", float(pro_q75))
print("Prolif Q25:", float(pro_q25))

# Initialize label
adata_cd8.obs["Tex_class"] = "Unlabelled"

# Apply labels
is_exh_hi = adata_cd8.obs["exhaustion_score_z"] >= exh_q75
is_pro_hi = adata_cd8.obs["prolif_score_z"] >= pro_q75
is_pro_lo = adata_cd8.obs["prolif_score_z"] <= pro_q25

adata_cd8.obs.loc[is_exh_hi & is_pro_hi, "Tex_class"] = "Prolif-Tex"
adata_cd8.obs.loc[is_exh_hi & is_pro_lo, "Tex_class"] = "Non-Prolif-Tex"

print("\nTex_class counts:")
print(adata_cd8.obs["Tex_class"].value_counts())

# Plot on UMAP
sc.pl.umap(adata_cd8, color=["Tex_class", "timepoint", "response"], wspace=0.4)

In [ ]:
# -----------------------------
# Block 18 — Therapy-linked summaries (pre/post, RE/NR) + patient-level summaries
#   Focus: proportion shifts rather than new clusters
# -----------------------------

# A) Cell-level group proportions (descriptive)
group_frac = (
    adata_cd8.obs
    .groupby(["timepoint", "response"])["Tex_class"]
    .value_counts(normalize=True)
    .rename("fraction")
    .reset_index()
)

print("\nFractions of Tex_class by timepoint × response:")
print(group_frac)

# B) Patient-level proportions (more thesis-defensible)
#    Compute per patient per timepoint the fraction of each class
pt_counts = (
    adata_cd8.obs
    .groupby(["patient", "timepoint", "response", "Tex_class"])
    .size()
    .rename("n_cells")
    .reset_index()
)

pt_total = (
    adata_cd8.obs
    .groupby(["patient", "timepoint", "response"])
    .size()
    .rename("n_total")
    .reset_index()
)

pt = pt_counts.merge(pt_total, on=["patient", "timepoint", "response"], how="left")
pt["frac"] = pt["n_cells"] / pt["n_total"]

print("\nPatient-level class fractions (head):")
print(pt.head(20))

# Optional: simplify to just Prolif-Tex vs Non-Prolif-Tex fractions
pt_pivot = pt.pivot_table(
    index=["patient", "timepoint", "response"],
    columns="Tex_class",
    values="frac",
    fill_value=0.0
).reset_index()

print("\nPatient-level pivot (head):")
print(pt_pivot.head(10))

In [ ]:
adata_cd8.obs["Tex_class"].value_counts(normalize=True)

In [ ]:
adata_cd8.obs.groupby(["timepoint", "response"])["Tex_class"].value_counts(normalize=True)

In [ ]:
pt_pivot.groupby(["timepoint", "response"], observed=True).mean(numeric_only=True)

In [ ]:
pt_pivot.groupby("patient")["timepoint"].nunique().value_counts()

In [ ]:
wide = pt_pivot.pivot_table(
    index=["patient", "response"],
    columns="timepoint",
    values=["Prolif-Tex", "Non-Prolif-Tex"],
    fill_value=0.0
)

wide["dProlifTex"] = wide[("Prolif-Tex", "post")] - wide[("Prolif-Tex", "pre")]
wide["dNonProlifTex"] = wide[("Non-Prolif-Tex", "post")] - wide[("Non-Prolif-Tex", "pre")]

wide[["dProlifTex", "dNonProlifTex"]].reset_index().head()

In [ ]:
# Make delta_df clean (flat columns)
delta_df = wide[["dProlifTex", "dNonProlifTex"]].reset_index()

# Flatten columns if any are tuples / MultiIndex-like
delta_df.columns = [
    "_".join(map(str, c)).strip("_") if isinstance(c, tuple) else str(c)
    for c in delta_df.columns
]

print(delta_df.columns.tolist())

In [ ]:
delta_df.groupby("response")[["dProlifTex", "dNonProlifTex"]].describe()

In [ ]:
from scipy.stats import mannwhitneyu

re = delta_df.loc[delta_df["response"]=="RE", "dProlifTex"].dropna()
nr = delta_df.loc[delta_df["response"]=="NR", "dProlifTex"].dropna()

u, p = mannwhitneyu(re, nr, alternative="two-sided")
print("Mann–Whitney U p =", p)
print("RE median:", float(re.median()), "NR median:", float(nr.median()))

In [ ]:
import numpy as np
def cliffs_delta(x, y):
    x = np.asarray(x); y = np.asarray(y)
    return (np.sum(x[:,None] > y[None,:]) - np.sum(x[:,None] < y[None,:])) / (len(x)*len(y))

print("Cliff's delta:", cliffs_delta(re.values, nr.values))

In [ ]:
import matplotlib.pyplot as plt

delta_df.boxplot(column="dProlifTex", by="response")
plt.suptitle("")
plt.title("Δ Prolif-Tex fraction (post − pre) by response")
plt.ylabel("Δ fraction")
plt.show()

In [ ]:
for resp in ["RE","NR"]:
    x = delta_df.loc[delta_df["response"]==resp, "dProlifTex"].values
    print(resp, "n=", len(x), "mean=", x.mean(), "median=", np.median(x))

In [ ]:
# continuous proliferation score delta
score_wide = adata_cd8.obs.pivot_table(
    index=["patient","response"],
    columns="timepoint",
    values="prolif_score_z"
)

score_wide["dProlifScore"] = score_wide["post"] - score_wide["pre"]

In [ ]:
delta_cont = score_wide.reset_index()

from scipy.stats import mannwhitneyu

re = delta_cont.loc[delta_cont["response"]=="RE", "dProlifScore"]
nr = delta_cont.loc[delta_cont["response"]=="NR", "dProlifScore"]

u, p = mannwhitneyu(re, nr, alternative="two-sided")

print("Mann–Whitney p =", p)
print("RE median:", float(re.median()), "NR median:", float(nr.median()))

In [ ]:
import numpy as np

def cliffs_delta(x, y):
    x = np.asarray(x); y = np.asarray(y)
    return (np.sum(x[:,None] > y[None,:]) - np.sum(x[:,None] < y[None,:])) / (len(x)*len(y))

print("Cliff's delta:", cliffs_delta(re.values, nr.values))

In [ ]:
import matplotlib.pyplot as plt

delta_cont.boxplot(column="dProlifScore", by="response")
plt.suptitle("")
plt.title("Δ Continuous Proliferation Score (post − pre)")
plt.ylabel("Δ z-score")
plt.show()

In [ ]:
exh_wide = adata_cd8.obs.pivot_table(
    index=["patient","response"],
    columns="timepoint",
    values="exhaustion_score_z"
).dropna()

exh_wide["dExhScore"] = exh_wide["post"] - exh_wide["pre"]

delta_cont = delta_cont.merge(
    exh_wide[["dExhScore"]],
    left_on=["patient","response"],
    right_index=True
)

In [ ]:
delta_cont.groupby("response")[["dProlifScore","dExhScore"]].mean()

In [ ]:
for resp in ["RE","NR"]:
    sub = delta_cont[delta_cont["response"]==resp]
    plt.scatter(sub["dExhScore"], sub["dProlifScore"], label=resp)

plt.axhline(0,color="gray",linestyle="--")
plt.axvline(0,color="gray",linestyle="--")
plt.xlabel("Δ Exhaustion Score")
plt.ylabel("Δ Proliferation Score")
plt.legend()
plt.title("Patient-level movement in exhaustion–proliferation space")
plt.show()

In [ ]:
from scipy.stats import spearmanr

rho, p_corr = spearmanr(delta_cont["dExhScore"], delta_cont["dProlifScore"])
print("Spearman rho:", rho, "p =", p_corr)

In [ ]:
delta_cont = score_wide.reset_index()
print(delta_cont["dProlifScore"].isna().sum(), "NaNs in dProlifScore")
print(delta_cont.groupby("response")["dProlifScore"].apply(lambda x: x.isna().sum()))
print(delta_cont.groupby("response")["dProlifScore"].count())

In [ ]:
df = adata_cd8.obs[["patient", "response", "timepoint", "prolif_score_z", "exhaustion_score_z"]].copy()
df = df.dropna(subset=["patient", "response", "timepoint", "prolif_score_z", "exhaustion_score_z"])

In [ ]:
pt_scores = (
    df.groupby(["patient", "response", "timepoint"], observed=True)
      [["prolif_score_z", "exhaustion_score_z"]]
      .mean()
      .reset_index()
)

print(pt_scores.head())

In [ ]:
pt_wide = pt_scores.pivot_table(
    index=["patient", "response"],
    columns="timepoint",
    values=["prolif_score_z", "exhaustion_score_z"]
)

# Keep only paired patients
pt_wide = pt_wide.dropna(subset=[("prolif_score_z","pre"), ("prolif_score_z","post"),
                                 ("exhaustion_score_z","pre"), ("exhaustion_score_z","post")])

pt_wide["dProlifScore"] = pt_wide[("prolif_score_z","post")] - pt_wide[("prolif_score_z","pre")]
pt_wide["dExhScore"]    = pt_wide[("exhaustion_score_z","post")] - pt_wide[("exhaustion_score_z","pre")]

delta_pt = pt_wide[["dProlifScore","dExhScore"]].reset_index()
delta_pt.head()

In [ ]:
from scipy.stats import mannwhitneyu
import numpy as np

re = delta_pt.loc[delta_pt["response"]=="RE", "dProlifScore"].dropna()
nr = delta_pt.loc[delta_pt["response"]=="NR", "dProlifScore"].dropna()

u, p = mannwhitneyu(re, nr, alternative="two-sided")
print("Mann–Whitney p =", p)
print("RE median:", float(re.median()), "NR median:", float(nr.median()), "RE n:", len(re), "NR n:", len(nr))

def cliffs_delta(x, y):
    x = np.asarray(x); y = np.asarray(y)
    return (np.sum(x[:,None] > y[None,:]) - np.sum(x[:,None] < y[None,:])) / (len(x)*len(y))

print("Cliff's delta:", cliffs_delta(re.values, nr.values))

In [ ]:
import matplotlib.pyplot as plt

for resp in ["RE", "NR"]:
    sub = delta_pt[delta_pt["response"]==resp]
    plt.scatter(sub["dExhScore"], sub["dProlifScore"], label=resp)

plt.axhline(0, color="gray", linestyle="--")
plt.axvline(0, color="gray", linestyle="--")
plt.xlabel("Δ Exhaustion Score (patient mean post − pre)")
plt.ylabel("Δ Proliferation Score (patient mean post − pre)")
plt.legend()
plt.title("Patient-level movement in exhaustion–proliferation space")
plt.show()

In [ ]:
pt_tail = (
    df.groupby(["patient","response","timepoint"], observed=True)
      .agg(
          prolif_p90=("prolif_score_z", lambda x: np.quantile(x, 0.90))
      )
      .reset_index()
)

tail_wide = pt_tail.pivot_table(
    index=["patient","response"],
    columns="timepoint",
    values="prolif_p90"
).dropna()

tail_wide["dProlifP90"] = tail_wide["post"] - tail_wide["pre"]

delta_tail = tail_wide[["dProlifP90"]].reset_index()

re = delta_tail.loc[delta_tail["response"]=="RE", "dProlifP90"]
nr = delta_tail.loc[delta_tail["response"]=="NR", "dProlifP90"]

from scipy.stats import mannwhitneyu
u,p = mannwhitneyu(re,nr)
print("MW p:", p)
print("RE median:", float(re.median()), "NR median:", float(nr.median()))
print("Cliff:", cliffs_delta(re.values,nr.values))

In [ ]:
baseline = pt_tail[pt_tail["timepoint"]=="pre"]

baseline_p90 = baseline.pivot_table(
    index=["patient","response"],
    values="prolif_p90"
).reset_index()

baseline_p90.groupby("response")["prolif_p90"].describe()

In [ ]:
adata_cd8.obs.groupby(["pat_clone_ID","timepoint"]).size()

In [ ]:
# -----------------------------
# Block 19 — Save CD8-only object (optional but recommended)
# -----------------------------
adata_cd8.write_h5ad(out_cd8_h5ad)
print("\nSaved CD8-only h5ad:", out_cd8_h5ad)